# Recommendation 04 — Fase 2A: LLM → profilo semantico → GTE

**Domanda sperimentale:** a parità di rating visibili, la descrizione semantica generata da un LLM migliora il retrieval rispetto al **GTE diretto** del notebook 02?

- Stesso protocollo `rs_coldstart_v1`, stesso catalogo, stessi livelli `k = 0,1,3,5,10` e medesimi 20 target **A**; valutazione **B** con tutti i rating non rivelati.
- Il profilo LLM vede **solo rating rivelati**, titolo e generi *pubblici* da `movies.csv`. Non vede nessun file `private_evaluation` e non usa i tag MovieLens come input.
- Output LLM strutturato: query positiva e negativa in inglese. Scoring **con le stesse regole della baseline GTE** (alfa negativa selezionata nel notebook 02, popularity fallback con soli negativi/intermedi). Nessun reranker, critic, chiarimento, answer agent o hard genre filter.
- Gli embedding dei **film** NON vengono ricalcolati. Da Qdrant (`resource_id == movieId`) si caricano gli esistenti; si calcolano soltanto embedding delle **nuove query** LLM con lo stesso modello GTE.
- L'esecuzione iniziale è un **pilota validation** di 10 utenti × 1 replica, al massimo 40 chiamate; la selezione di prompt e metodo si fa soltanto sulla validation. Il test è protetto da una conferma esplicita.

**Nota di riproducibilità:** il notebook usa gli adapter `movie_rag.llm_config` / `llm_providers` per chiamare l'LLM, ma è un esperimento separato dal workflow completo, che è oggetto delle fasi 2B–3. Se l'API del progetto è stata cambiata dopo il bundle di settembre, aggiornare il solo adapter di questa fase, senza usare una versione vecchia della `src/`.

## 0. Configurazione

Inserisci questo notebook sotto `<progetto>/notebooks/`. La configurazione predefinita usa l'ambiente corrente del progetto e Qdrant locale. **Le chiamate LLM possono avere un costo**: il pilota è volutamente piccolo. Non vengono mai sovrascritti i file dei notebook 01–03.

In [1]:
from pathlib import Path
import os, sys, json, hashlib, asyncio, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT_OVERRIDE = None     # es. Path('/percorso/al/progetto')
PROTOCOL_ID = 'rs_coldstart_v1'
RUN_MODE = 'pilot_validation'    # 'pilot_validation', 'full_validation', 'test'
ALLOW_TEST_RUN = False           # richiede True + RUN_MODE='test' (il test originale è già stato ispezionato)
PILOT_USERS = 10                # utenti validation distinti
PILOT_REPLICA = 0               # mantiene una sola replica per utente nel pilota
K_LEVELS = (0, 1, 3, 5, 10)
TOP_K = 10
POS_MIN = 4.0
NEG_MAX = 2.0
MAX_NEW_LLM_CALLS = 40          # limite di spesa, alzare esplicitamente per full_validation/test
SLEEP_BETWEEN_CALLS = 0.0
MAX_ATTEMPTS_PER_LLM_CALL = 2

# LLM: usa configurazioni/provider REALI del package movie_rag già presente, senza una seconda integrazione SDK.
LLM_CONFIG_PATH = 'configs/llm_profiles_v1.json'
LLM_PROFILE_OVERRIDE = None     # None = profilo associato al Planner nel workflow configurato
PROMPT_VERSION = 'rec04_semantic_profile_v1'

# Embedding GTE: stessa collection del notebook 02, resource_id è il movieId MovieLens.
QDRANT_URL = 'http://localhost:6333'
QDRANT_COLLECTION = 'movies_d_full_enriched_v2_gte_multilingual_base_v1'
QDRANT_MOVIE_ID_KEY = 'resource_id'
QDRANT_VECTOR_NAME = None
QDRANT_API_KEY = None           # per chiave remota usare variabile ambiente QDRANT_API_KEY
GTE_MODEL_ID = 'Alibaba-NLP/gte-multilingual-base'  # query encoder; dimensione attesa 768
GTE_MODEL_TRUST_REMOTE_CODE = True
GTE_QUERY_BATCH_SIZE = 16
RANDOM_SEED = 20261021


def find_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (folder/'datasets/processed/recommendation_protocol'/PROTOCOL_ID/'protocol_manifest.json').is_file():
            return folder
    raise FileNotFoundError('Protocollo non trovato: imposta PROJECT_ROOT_OVERRIDE')

ROOT = find_root()
BASE = ROOT/'datasets/processed/recommendation_protocol'/PROTOCOL_ID
INPUT = BASE/'inputs'
RESULTS = BASE/'evaluations'/'llm_gte_phase2a'
PRED = BASE/'predictions'
MOVIES_CSV = ROOT/'datasets/raw/ml-latest-small/movies.csv'
if RUN_MODE not in ('pilot_validation','full_validation','test'):
    raise ValueError('RUN_MODE non valido')
if RUN_MODE == 'test' and not ALLOW_TEST_RUN:
    raise RuntimeError('Test disabilitato: fissare il metodo prima, poi impostare ALLOW_TEST_RUN=True')
print('ROOT:',ROOT,'| MODE:',RUN_MODE,'| destinazione:',RESULTS/RUN_MODE)

ROOT: /Users/danieleraveglia/Documents/uniMi/TESI/progetto | MODE: pilot_validation | destinazione: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1/evaluations/llm_gte_phase2a/pilot_validation


## 1. Input ammessi all'LLM: soltanto interazioni visibili

Il file `movies.csv` serve esclusivamente per **titolo e generi**. Non passiamo statistiche MovieLens aggregate, rating del test, `private_profile`, `private_fixed_gt`, `private_additional` o descrizioni che incorporino user tag.

In [2]:
manifest = json.loads((BASE/'protocol_manifest.json').read_text(encoding='utf-8'))
assert manifest['protocol_id'] == PROTOCOL_ID
train = pd.read_csv(INPUT/'train_ratings.csv')
movies = pd.read_csv(MOVIES_CSV)
catalog = pd.read_csv(INPUT/'catalog_movie_ids.csv')
scenarios = pd.read_csv(INPUT/'scenarios.csv')
levels = pd.read_csv(INPUT/'scenario_levels.csv')
visible = pd.read_csv(INPUT/'visible_interactions.csv')
for f in (train,movies,catalog,visible):
    f['movieId'] = f['movieId'].astype('int64')
for f in (visible,levels):
    f['k'] = f['k'].astype(int)
if catalog.movieId.duplicated().any(): raise ValueError('Film duplicati nel catalogo')
ids = np.sort(catalog.movieId.to_numpy(dtype=np.int64))
idx_movie = {int(m):i for i,m in enumerate(ids)}
assert set(ids).issubset(set(movies.movieId))
assert not set(train.userId).intersection(set(scenarios.userId))
title_by_id = movies.set_index('movieId')[['title','genres']].to_dict('index')
visible_by_level = {(str(sid),int(k)):grp[['movieId','rating']].copy()
                    for (sid,k),grp in visible.groupby(['scenario_id','k'],sort=False)}
selected_split = 'test' if RUN_MODE=='test' else 'validation'
selected_scenarios = scenarios.loc[scenarios.split==selected_split].copy()
if RUN_MODE=='pilot_validation':
    # Una replica e N utenti scelti con ordine deterministico indipendente dai rating
    selected_scenarios = selected_scenarios.loc[
        selected_scenarios.scenario_id.astype(str).str.endswith(f'_r{PILOT_REPLICA}')].copy()
    chosen_users = np.random.default_rng(RANDOM_SEED).choice(
        np.sort(selected_scenarios.userId.unique()),
        size=min(PILOT_USERS,selected_scenarios.userId.nunique()),replace=False)
    selected_scenarios = selected_scenarios.loc[selected_scenarios.userId.isin(chosen_users)]
selected = levels.loc[levels.scenario_id.isin(set(selected_scenarios.scenario_id)) &
                      levels.k.isin(K_LEVELS)].sort_values(['scenario_id','k']).reset_index(drop=True)
assert len(selected)==len(selected_scenarios)*len(K_LEVELS)
for rec in selected.itertuples(index=False):
    obs=visible_by_level.get((str(rec.scenario_id),int(rec.k)),pd.DataFrame(columns=['movieId','rating']))
    assert len(obs)==int(rec.k) and obs.movieId.is_unique
    assert set(obs.movieId).issubset(set(ids))
print('Utenti:',selected_scenarios.userId.nunique(),'| scenari:',len(selected_scenarios),
      '| livelli:',len(selected),'| chiamate LLM massime (non inclusi fallback):',len(selected_scenarios)*4)
display(selected.groupby('k').size().rename('scenari').to_frame().T)

Utenti: 10 | scenari: 10 | livelli: 50 | chiamate LLM massime (non inclusi fallback): 40


k,0,1,3,5,10
scenari,10,10,10,10,10


## 2. Modelli di confronto e punteggi fissi

Usiamo la **stessa Popularity smoothed** del notebook 02 e il peso negativo GTE salvato da quella validazione. Per il confronto appaiato saranno riletti i ranking *già prodotti* dal notebook 02. Non si riaddestra MF né si ricalcolano gli embedding del catalogo.

In [3]:
config_path=BASE/'evaluations'/'baseline_mf_gte'/'model_config.json'
if not config_path.is_file():
    raise FileNotFoundError(f'Configurazione del benchmark 02 mancante: {config_path}')
config=json.loads(config_path.read_text(encoding='utf-8'))
if config.get('protocol_id')!=PROTOCOL_ID or not config.get('gte_available'):
    raise ValueError('Config baseline non corrispondente al protocollo oppure GTE non disponibile')
ALPHA = float(config['gte_neg_alpha_selected'])
NEG_ONLY_ALPHA = float(config['gte_negative_only_alpha'])
POPULARITY_PRIOR = float(config['popularity_prior'])
mu=float(train.rating.mean())
agg=train.groupby('movieId').rating.agg(['sum','count']).reindex(ids,fill_value=0)
popularity=(agg['sum'].to_numpy(float)+POPULARITY_PRIOR*mu)/(agg['count'].to_numpy(float)+POPULARITY_PRIOR)
pop_z=(popularity-popularity.mean())/max(float(popularity.std()),1e-8)
print('GTE negative α:',ALPHA,'| GTE negative-only α:',NEG_ONLY_ALPHA,'| prior:',POPULARITY_PRIOR)

base_file=PRED/('rankings_baselines_test.csv' if selected_split=='test' else 'rankings_baselines_validation.csv')
if not base_file.is_file(): raise FileNotFoundError(f'Ranking baseline 02 mancante: {base_file}')
base_rank=pd.read_csv(base_file)
if selected_split=='validation':
    gte_base_name=f'gte_negalpha_{ALPHA:g}'
else:
    gte_base_name='gte_direct'
assert gte_base_name in set(base_rank.model), f'Modello GTE {gte_base_name} non presente: {list(base_rank.model.unique())}'
base_rank=base_rank.loc[base_rank.scenario_id.isin(set(selected.scenario_id)) & base_rank.k.isin(K_LEVELS)].copy()
base_rank.loc[base_rank.model==gte_base_name,'model']='gte_direct'
print('Baseline disponibili:',sorted(base_rank.model.unique()))

GTE negative α: 0.2 | GTE negative-only α: 0.35 | prior: 15.0
Baseline disponibili: ['gte_direct', 'gte_negalpha_0.5', 'gte_negalpha_0.8', 'mf_foldreg_1', 'mf_foldreg_20', 'mf_foldreg_5', 'popularity']


## 3. Carica il catalogo vettoriale GTE **già indicizzato**

`resource_id` è il `movieId` MovieLens; **non** `tmdb_id`. Richiediamo copertura **100%**, assenza di duplicati e dimensione coerente con il query encoder. Non modifichiamo Qdrant.

In [4]:
from qdrant_client import QdrantClient
client_kwargs={'url':QDRANT_URL,'timeout':60}
if QDRANT_API_KEY or os.getenv('QDRANT_API_KEY'):
    client_kwargs['api_key']=QDRANT_API_KEY or os.getenv('QDRANT_API_KEY')
client=QdrantClient(**client_kwargs)
assert QDRANT_COLLECTION in [c.name for c in client.get_collections().collections], 'Collection GTE mancante'

vec_by_id={}; offset=None; sample_doc=None
while True:
    points, offset = client.scroll(collection_name=QDRANT_COLLECTION,offset=offset,limit=256,
                                   with_payload=True,with_vectors=True)
    for pt in points:
        payload=pt.payload or {}
        if QDRANT_MOVIE_ID_KEY not in payload:
            raise KeyError(f'Payload Qdrant senza {QDRANT_MOVIE_ID_KEY}: {list(payload)}')
        mid=int(payload[QDRANT_MOVIE_ID_KEY])
        if mid in vec_by_id: raise ValueError(f'Duplicato movieId={mid}')
        vec=pt.vector
        if isinstance(vec,dict):
            vec_name=QDRANT_VECTOR_NAME or (next(iter(vec)) if len(vec)==1 else None)
            if vec_name is None:raise ValueError(f'Qdrant vectors: specificare nome fra {list(vec)}')
            vec=vec[vec_name]
        if vec is None:raise ValueError(f'Vettore mancante per film {mid}')
        vec_by_id[mid]=np.asarray(vec,dtype=np.float32)
        if sample_doc is None and isinstance(payload.get('text'),str):sample_doc=(mid,payload['text'])
    if offset is None:break
missing=sorted(set(ids)-set(vec_by_id))
if missing: raise ValueError(f'Copertura insufficiente, film mancanti: {missing[:20]}')
gte=np.stack([vec_by_id[int(mid)] for mid in ids]).astype(np.float32)
if not np.isfinite(gte).all():raise ValueError('GTE NaN/Inf')
norms=np.linalg.norm(gte,axis=1)
if (norms<1e-9).any():raise ValueError('GTE vettore nullo')
gte=gte/norms[:,None]
assert gte.shape[1]==768, f'Attesi 768 dim GTE multilingual base, trovati {gte.shape[1]}'
print('Qdrant:',QDRANT_COLLECTION,'| film:',len(ids),'| dim:',gte.shape[1],
      '| esempio:',sample_doc[0] if sample_doc else 'nessuno')

Qdrant: movies_d_full_enriched_v2_gte_multilingual_base_v1 | film: 1996 | dim: 768 | esempio: 1


## 4. Profilo semantico LLM: contratto strutturato

Nessuna simulazione utente in fase 2A: l'LLM vede il **medesimo** profilo `k` che vede il GTE diretto. Genera due stringhe di retrieval (positive/negative) senza suggerire film o usare dati nascosti. Gli output sono conservati in cache per riproducibilità.

In [6]:
from pydantic import BaseModel, ConfigDict, Field
from typing import Optional

class SemanticProfile(BaseModel):
    model_config = ConfigDict(extra='forbid')
    positive_query: Optional[str] = Field(default=None, max_length=600)
    negative_query: Optional[str] = Field(default=None, max_length=600)
    brief_rationale: str = Field(max_length=350)

SYSTEM_PROMPT = """You are a deterministic movie-preference query writer for a recommender experiment.
You receive ONLY movie titles, public genres and the user's explicitly revealed numerical ratings.
Ground all preferences in these observations. Do not invent unseen ratings, personal background,
extra titles, hidden interests, or specific facts about scenes/plots not provided.
Produce compact ENGLISH semantic descriptions usable as separate dense retrieval queries.
- positive_query: themes, style and genres of positively rated movies (>=4), expressed positively.
- negative_query: themes, style and genres of negatively rated movies (<=2), phrased as a description
  of UNWANTED content (not as an imperative instruction and not with the word 'avoid').
- Null a query when that class of feedback is absent. Ratings 2.5–3.5 are neutral: they must
  not be converted into liked/disliked content.
- Do not mention technical scoring, MovieLens, or evaluation.
- Do not overgeneralize from one movie. Never include movie recommendations.
Return only the defined structured fields."""

def observed_payload(obs):
    return [{'movie_id':int(r.movieId),'title':str(title_by_id[int(r.movieId)]['title']),
             'genres':str(title_by_id[int(r.movieId)]['genres']).replace('|',', '),
             'rating':float(r.rating)} for r in obs.sort_values('movieId').itertuples(index=False)]

def observed_signature(obs):
    src=json.dumps(observed_payload(obs),sort_keys=True,ensure_ascii=False)
    return hashlib.sha256(src.encode('utf-8')).hexdigest()

PROMPT_HASH=hashlib.sha256((PROMPT_VERSION+'\n'+SYSTEM_PROMPT).encode('utf-8')).hexdigest()

# Check rapido: dati profilo derivati SOLO da visible_interactions.csv + movies.csv.
example = next((x for x in selected.itertuples(index=False) if x.k>0),None)
if example:
    display(pd.DataFrame(observed_payload(visible_by_level[(example.scenario_id,int(example.k))])))

,movie_id,title,genres,rating
0,17,Sense and Sensibility (1995),"Drama, Romance",4.5


## 5. Adapter LLM del progetto e cache incrementale

Il provider viene da `movie_rag.llm_providers`, con profilo versionato `configs/llm_profiles_v1.json`. La cache include hash degli input, prompt, provider e modello; **nessun errore API viene sostituito silenziosamente da Popularity**. Con `RUN_MODE='test'` non vengono letti i dati di test per adattare il prompt.

In [ ]:
# Nessun bundle legacy copiato dentro src/: si importa il package ATTUALE installato nel progetto.
SRC=ROOT/'src'
if str(SRC) not in sys.path: sys.path.insert(0,str(SRC))
from movie_rag.llm_config import load_llm_config, role_profile_overrides_from_environment
from movie_rag.llm_providers import LLMProviderRegistry

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT/'.env',override=False)
except ImportError:
    print('python-dotenv non disponibile; assumo API key già esportata in ambiente')

llm_catalog=load_llm_config(ROOT/LLM_CONFIG_PATH)
llm_workflow=llm_catalog.get_workflow()
planner=llm_catalog.get_agent(llm_workflow.planner_agent)
overrides=role_profile_overrides_from_environment()
selected_profile_id=LLM_PROFILE_OVERRIDE or overrides.get('planner') or planner.llm_profile
llm_profile=llm_catalog.get_llm_profile(selected_profile_id)
provider=LLMProviderRegistry(llm_catalog).get(selected_profile_id)
print('LLM provider:',llm_profile.provider,'| modello:',llm_profile.model,'| profilo:',selected_profile_id)

RUN_DIR=RESULTS/RUN_MODE
RUN_DIR.mkdir(parents=True,exist_ok=True)
CACHE_PATH=RESULTS/'semantic_profiles_cache.jsonl'  # cache condivisa fra pilot/full/test, keyed su scenario+input+modello+prompt
PROFILE_MODEL=f'{llm_profile.provider}:{llm_profile.model}:{selected_profile_id}'

cache={}
if CACHE_PATH.exists():
    for line in CACHE_PATH.read_text(encoding='utf-8').splitlines():
        if not line.strip():continue
        row=json.loads(line)
        cache[(row['scenario_id'],int(row['k']),row['input_hash'],row['prompt_hash'],row['profile_model'])]=row
print('Profili presenti nella cache:',len(cache))

async def make_profile(sid,k,obs):
    key=(sid,k,observed_signature(obs),PROMPT_HASH,PROFILE_MODEL)
    if key in cache:return cache[key],False
    pos=int((obs.rating>=POS_MIN).sum())
    neg=int((obs.rating<=NEG_MAX).sum())
    if pos==0 and neg==0:
        profile=SemanticProfile(positive_query=None,negative_query=None,brief_rationale='Fallback deterministico: nessun feedback non neutro')
        trace=None
    else:
        prompt=json.dumps({'task':'represent_observed_movie_preferences',
                           'positive_rating_threshold':POS_MIN,'negative_rating_threshold':NEG_MAX,
                           'revealed_movies':observed_payload(obs)},ensure_ascii=False,sort_keys=True)
        last_exc=None
        for attempt in range(MAX_ATTEMPTS_PER_LLM_CALL):
            try:
                generation=await provider.generate_structured(
                    agent='recommendation_2a_semantic_profile',prompt_version=PROMPT_VERSION,
                    system_prompt=SYSTEM_PROMPT,user_prompt=prompt,output_schema=SemanticProfile,
                    temperature=0.0,max_output_tokens=700)
                profile=generation.value
                trace=generation.trace.model_dump(mode='json')
                break
            except Exception as exc:
                last_exc=exc
                if attempt+1<MAX_ATTEMPTS_PER_LLM_CALL: await asyncio.sleep(2**attempt)
        else:
            raise RuntimeError(f'LLM non riuscito: scenario={sid} k={k}') from last_exc
        if bool(profile.positive_query and profile.positive_query.strip()) != bool(pos):
            raise ValueError(f'LLM positive_query incoerente con rating: scenario={sid} k={k}')
        if bool(profile.negative_query and profile.negative_query.strip()) != bool(neg):
            raise ValueError(f'LLM negative_query incoerente con rating: scenario={sid} k={k}')
    row={'scenario_id':sid,'k':int(k),'input_hash':key[2],'prompt_hash':PROMPT_HASH,
         'profile_model':PROFILE_MODEL,**profile.model_dump(), 'trace':trace}
    # Scrittura immediata per resumability. I file sono confinati nella directory di QUESTO run.
    with CACHE_PATH.open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,ensure_ascii=False)+'\n')
    cache[key]=row
    return row, bool(trace)

async def produce_all_profiles():
    required=[]
    for row in selected.itertuples(index=False):
        sid,k=str(row.scenario_id),int(row.k)
        obs=visible_by_level.get((sid,k),pd.DataFrame(columns=['movieId','rating']))
        key=(sid,k,observed_signature(obs),PROMPT_HASH,PROFILE_MODEL)
        needs_llm=bool(((obs.rating>=POS_MIN)|(obs.rating<=NEG_MAX)).any())
        if key not in cache and needs_llm:required.append((sid,k))
    if len(required)>MAX_NEW_LLM_CALLS:
        raise RuntimeError(f'Servirebbero {len(required)} nuove chiamate > MAX_NEW_LLM_CALLS={MAX_NEW_LLM_CALLS}. '
                           'Aumenta il limite consapevolmente oppure riduci la selezione.')
    completed=[];calls=0
    for row in selected.itertuples(index=False):
        sid,k=str(row.scenario_id),int(row.k)
        obs=visible_by_level.get((sid,k),pd.DataFrame(columns=['movieId','rating']))
        profile,called=await make_profile(sid,k,obs)
        completed.append(profile)
        calls+=int(called)
        if called and SLEEP_BETWEEN_CALLS: await asyncio.sleep(SLEEP_BETWEEN_CALLS)
        if len(completed)%10==0: print('Completati profili:',len(completed),'/',len(selected),'| nuove chiamate:',calls)
    return pd.DataFrame(completed)

profiles=await produce_all_profiles()
assert len(profiles)==len(selected)
for p in profiles.to_dict('records'):
    if (p.trace != None):
        obs=visible_by_level.get((str(p['scenario_id']),int(p['k'])),pd.DataFrame(columns=['movieId','rating']))
        expected_pos=bool((obs.rating>=POS_MIN).any())
        expected_neg=bool((obs.rating<=NEG_MAX).any())
        assert bool(p.get('positive_query') and str(p['positive_query']).strip())==expected_pos
        assert bool(p.get('negative_query') and str(p['negative_query']).strip())==expected_neg
print('Profili pronti:',len(profiles),'| query positive:',profiles.positive_query.notna().sum(),
      '| query negative:',profiles.negative_query.notna().sum())
display(profiles[['scenario_id','k','positive_query','negative_query']].head(8))

LLM provider: gemini | modello: gemini-3.5-flash-lite | profilo: gemini_free
Profili presenti nella cache: 21
Completati profili: 10 / 50 | nuove chiamate: 0
Completati profili: 20 / 50 | nuove chiamate: 0
Completati profili: 30 / 50 | nuove chiamate: 8
Completati profili: 40 / 50 | nuove chiamate: 15
Completati profili: 50 / 50 | nuove chiamate: 23


AssertionError: 

In [55]:
for p in profiles.to_dict('records'):
    if (p['trace'] != None):
        obs=visible_by_level.get((str(p['scenario_id']),int(p['k'])),pd.DataFrame(columns=['movieId','rating']))
        expected_pos=bool((obs.rating>=POS_MIN).any())
        expected_neg=bool((obs.rating<=NEG_MAX).any())
        # attenzione che per k = 0 non ci sono query positive o negative, quindi il campo può essere None o NaN o stringa vuota
        # assert bool(p.get('positive_query') and str(p['positive_query']).strip())==expected_pos
        assert bool(p.get('positive_query') and str(p.get('positive_query'))!= 'nan' and str(p['positive_query']).strip())==expected_pos
        assert bool(p.get('negative_query') and str(p.get('negative_query'))!= 'nan' and str(p['negative_query']).strip())==expected_neg
print('Profili pronti:',len(profiles),'| query positive:',profiles.positive_query.notna().sum(),
      '| query negative:',profiles.negative_query.notna().sum())
display(profiles[['scenario_id','k','positive_query','negative_query']].head(8))

Profili pronti: 50 | query positive: 37 | query negative: 13


,scenario_id,k,positive_query,negative_query
0,validation_u100_r0,0,NaN,NaN
1,validation_u100_r0,1,"Drama and romance themes, period pieces, and e...",NaN
2,validation_u100_r0,3,"drama, romance, comedy, period drama, emotiona...",NaN
3,validation_u100_r0,5,"drama, romance, emotional storytelling, coming...",NaN
4,validation_u100_r0,10,"Drama, romance, comedies and musicals, featuri...",Horror and thriller films involving intense su...
5,validation_u106_r0,0,NaN,NaN
6,validation_u106_r0,1,NaN,NaN
7,validation_u106_r0,3,epic adventure and fantasy films featuring mag...,NaN


In [54]:
p

{'scenario_id': 'validation_u100_r0',
 'k': 1,
 'input_hash': '62ce208a636d0a6d16cdc37ebd16044a99a5e1b9ab53ec22b664ff5c1f09d835',
 'prompt_hash': 'efdbe1515e2e5d9c86442f2851785bf148cb5790e05d48887285f37e8c1d96ba',
 'profile_model': 'gemini:gemini-3.5-flash-lite:gemini_free',
 'positive_query': 'Drama and romance themes, period pieces, and emotional character-driven stories',
 'negative_query': nan,
 'brief_rationale': 'The user provided one positive rating (4.5) for the drama and romance movie Sense and Sensibility (1995), so only a positive query is generated reflecting those genres and themes.',
 'trace': {'agent': 'recommendation_2a_semantic_profile',
  'provider': 'gemini',
  'profile_id': 'gemini_free',
  'model': 'gemini-3.5-flash-lite',
  'prompt_version': 'rec04_semantic_profile_v1',
  'started_at': '2026-10-27T11:22:10.650851Z',
  'duration_ms': 2364.877332933247,
  'usage': {'input_tokens': 309, 'output_tokens': 84, 'total_tokens': 393},
  'request_id': 'Y4ngarehHPXjkdUPnZGt4

## Valutazione tempistiche modelli di reranking:

#### Locale
| Reranker | secondi per 20 coppie | secondi a coppia
|---------|------|--------|
| `bge_reranker_v2_m3` | 817.20 s | 40.860 s/coppia |
| `qwen3_reranker_0.6b` | 899.30 s | 44.965 s/coppia | 
| `gte_multilingual_reranker` | 249.44 s | 12.472 s/coppia |

#### Colab
| Reranker | secondi per 20 coppie | secondi a coppia
|---------|------|--------|
| `bge_reranker_v2_m3` | 0.51 s | 0.026 s/coppia |
| `qwen3_reranker_0.6b` | 0.65 s | 0.032 s/coppia | 
| `gte_multilingual_reranker` | 0.16 s | 0.008 s/coppia |


In [23]:
profiles.to_dict('records')

[{'scenario_id': 'validation_u100_r0',
  'k': 0,
  'input_hash': '4f53cda18c2baa0c0354bb5f9a3ecbe5ed12ab4d8e11ba873c2f11161202b945',
  'prompt_hash': 'efdbe1515e2e5d9c86442f2851785bf148cb5790e05d48887285f37e8c1d96ba',
  'profile_model': 'gemini:gemini-3.5-flash-lite:gemini_free',
  'positive_query': nan,
  'negative_query': nan,
  'brief_rationale': 'Fallback deterministico: nessun feedback non neutro',
  'trace': None},
 {'scenario_id': 'validation_u100_r0',
  'k': 1,
  'input_hash': '62ce208a636d0a6d16cdc37ebd16044a99a5e1b9ab53ec22b664ff5c1f09d835',
  'prompt_hash': 'efdbe1515e2e5d9c86442f2851785bf148cb5790e05d48887285f37e8c1d96ba',
  'profile_model': 'gemini:gemini-3.5-flash-lite:gemini_free',
  'positive_query': 'Drama and romance themes, period pieces, and emotional character-driven stories',
  'negative_query': nan,
  'brief_rationale': 'The user provided one positive rating (4.5) for the drama and romance movie Sense and Sensibility (1995), so only a positive query is generated 

## 6. Codifica delle **sole query** semantiche con GTE

Carica `Alibaba-NLP/gte-multilingual-base` tramite `sentence-transformers` e codifica unicamente le query generate dall'LLM. I vettori del catalogo restano quelli preesistenti. Verifichiamo esplicitamente che dimensione e norma siano coerenti.

In [56]:
from sentence_transformers import SentenceTransformer

all_queries=sorted({str(q).strip() for field in ('positive_query','negative_query')
                    for q in profiles[field].dropna().tolist() if str(q).strip()})
query_vectors={}
if all_queries:
    encoder=SentenceTransformer(GTE_MODEL_ID, trust_remote_code=GTE_MODEL_TRUST_REMOTE_CODE)
    for start in range(0,len(all_queries),GTE_QUERY_BATCH_SIZE):
        batch=all_queries[start:start+GTE_QUERY_BATCH_SIZE]
        vv=np.asarray(encoder.encode(batch,normalize_embeddings=True,show_progress_bar=False),dtype=np.float32)
        assert vv.shape==(len(batch),gte.shape[1]),f'Incompatibilità query/corpus: {vv.shape} vs {gte.shape}'
        if not np.isfinite(vv).all():raise ValueError('Query embedding non finiti')
        query_vectors.update(zip(batch,vv))
    print('Codificate query uniche:',len(all_queries),'(nessun film ricodificato)')
else:print('Nessuna query da codificare: tutti gli scenari hanno fallback Popularity')

# Controllo *indicativo* della compatibilità dell'encoder, non un criterio di tuning:
# un documento Qdrant codificato con lo stesso encoder dovrebbe essere vicino al proprio vettore.
if all_queries and sample_doc and len(sample_doc[1])<10000:
    vdoc=np.asarray(encoder.encode([sample_doc[1]],normalize_embeddings=True,show_progress_bar=False)[0],dtype=np.float32)
    sim=float(vdoc@gte[idx_movie[sample_doc[0]]])
    print('Similarità self-check testo Qdrant / vettore salvato:',round(sim,4))
    if sim<0.75:
        warnings.warn('Self-check basso: verificare che GTE_MODEL_ID, preprocessing/pooling e collection coincidano!')

Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing NewModel: ['classifier.bias', 'classifier.weight']
- This IS expected if you are initializing NewModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing NewModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Codificate query uniche: 50 (nessun film ricodificato)
Similarità self-check testo Qdrant / vettore salvato: 1.0


## 7. Ranking LLM+GTE: formula congelata della baseline

Confronto controllato: positive_query → similarità coseno con il catalogo; negative_query → penalità con **α** già fissato. Se solo negativi, usa lo stesso fallback popularity-minus-negative del notebook 02. Se nessuna preferenza positiva/negativa (o `k=0`), Popularity. Si escludono **soltanto** i film rivelati.

In [57]:
def calculate_scores(obs,profile):
    pos=profile.get('positive_query')
    neg=profile.get('negative_query')
    pos=pos.strip() if isinstance(pos,str) and pos.strip() else None
    neg=neg.strip() if isinstance(neg,str) and neg.strip() else None
    if pos:
        score=gte@query_vectors[pos]
        if neg: score=score-ALPHA*(gte@query_vectors[neg])
        return score.astype(float)
    if neg:
        return (pop_z-NEG_ONLY_ALPHA*(gte@query_vectors[neg])*3.0).astype(float)
    return popularity.copy()

def topk(scores,obs):
    score=np.asarray(scores,dtype=float).copy()
    if score.shape!=(len(ids),) or not np.isfinite(score).all():
        raise ValueError('Score non validi')
    for mid in obs.movieId:score[idx_movie[int(mid)]]=-np.inf
    order=np.lexsort((ids,-score))
    order=order[np.isfinite(score[order])][:TOP_K]
    assert len(order)==TOP_K
    assert not set(ids[order]).intersection(set(obs.movieId))
    return [(int(ids[j]),int(rank),float(score[j])) for rank,j in enumerate(order,1)]

records=[]
for p in profiles.to_dict('records'):
    sid,k=str(p['scenario_id']),int(p['k'])
    obs=visible_by_level.get((sid,k),pd.DataFrame(columns=['movieId','rating']))
    for mid,rank,score in topk(calculate_scores(obs,p),obs):
        records.append({'scenario_id':sid,'k':k,'model':'llm_gte_semantic',
                        'movieId':mid,'rank':rank,'score':score})
llm_rank=pd.DataFrame(records)
assert len(llm_rank)==len(selected)*TOP_K
# k=0 DEVE riprodurre Popularity (fallback pari a notebook 02).
for sid in selected.loc[selected.k==0,'scenario_id']:
    a=llm_rank.loc[(llm_rank.scenario_id==sid)&(llm_rank.k==0)].sort_values('rank').movieId.tolist()
    b=base_rank.loc[(base_rank.scenario_id==sid)&(base_rank.k==0)&
                    (base_rank.model=='popularity')].sort_values('rank').movieId.tolist()
    if b: assert a==b, f'Il fallback k=0 differisce da Popularity per {sid}'
RUN_DIR.mkdir(parents=True,exist_ok=True)
RANKINGS_FILE=RUN_DIR/'rankings_llm_gte.csv'
llm_rank.to_csv(RANKINGS_FILE,index=False)
print('Ranking LLM salvati:',RANKINGS_FILE,'| film consigliati:',len(llm_rank))

Ranking LLM salvati: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1/evaluations/llm_gte_phase2a/pilot_validation/rankings_llm_gte.csv | film consigliati: 500


## 8. SOLO ORA: accesso privato al ground truth per valutazione A/B

Questa cella viene dopo il salvataggio dei ranking. La generazione non ha mai accesso a `private_evaluation`. Stesso comportamento del valutatore del notebook 02: A = 20 rating fissi, B = A + rating extra + profilo non ancora rivelato. Gli item senza rating non sono trattati come negativi osservati.

In [58]:
PRIVATE=BASE/'private_evaluation'
profile_private=pd.read_csv(PRIVATE/'private_profile.csv')
fixed_private=pd.read_csv(PRIVATE/'private_fixed_gt.csv')
extra_private=pd.read_csv(PRIVATE/'private_additional.csv')
profile_gt={str(s):g.sort_values('reveal_order') for s,g in profile_private.groupby('scenario_id')}
fixed_gt={str(s):dict(zip(g.movieId.astype(int),g.rating.astype(float))) for s,g in fixed_private.groupby('scenario_id')}
extra_gt={str(s):dict(zip(g.movieId.astype(int),g.rating.astype(float))) for s,g in extra_private.groupby('scenario_id')}
owner=dict(zip(scenarios.scenario_id,scenarios.userId))
scenario_split=dict(zip(scenarios.scenario_id,scenarios.split))
valid_keys=set(map(tuple,selected[['scenario_id','k']].itertuples(index=False,name=None)))

def evaluate(pred):
    results=[]
    for (sid,k,model),g in pred.groupby(['scenario_id','k','model'],sort=True):
        sid,k=str(sid),int(k)
        if (sid,k) not in valid_keys:raise ValueError(f'Livello non selezionato: {(sid,k)}')
        g=g.sort_values('rank')
        assert len(g)==TOP_K and g.movieId.is_unique
        assert np.array_equal(g['rank'].to_numpy(),np.arange(1,TOP_K+1))
        actual=list(g.movieId.astype(int))
        obs=visible_by_level.get((sid,k),pd.DataFrame(columns=['movieId']))
        assert not set(actual).intersection(set(obs.movieId))
        assert set(actual).issubset(idx_movie)
        hidden_profile=profile_gt[sid].loc[profile_gt[sid].reveal_order>k]
        for scheme in ('A','B'):
            truth=dict(fixed_gt[sid])
            if scheme=='B':
                truth.update(extra_gt.get(sid,{}))
                truth.update(dict(zip(hidden_profile.movieId.astype(int),hidden_profile.rating.astype(float))))
            pos={int(m) for m,rate in truth.items() if rate>=POS_MIN}
            labels=np.array([int(mid in pos) for mid in actual],dtype=float)
            npos=len(pos)
            if npos:
                weights=1/np.log2(np.arange(2,TOP_K+2))
                ideal=weights[:min(npos,TOP_K)].sum()
                ndcg=float((labels*weights).sum()/ideal)
                recall=float(labels.sum()/npos)
                hit=float(labels.sum()>0)
            else:ndcg=recall=hit=np.nan
            results.append({'scenario_id':sid,'userId':int(owner[sid]),'split':scenario_split[sid],
                            'k':k,'model':model,'scheme':scheme,'n_relevant':npos,
                            'n_hits_topk':int(labels.sum()),'ndcg@10':ndcg,
                            'recall@10':recall,'hr@10':hit,
                            'n_unjudged_topk':sum(mid not in truth for mid in actual)})
    return pd.DataFrame(results)

# Confronti soltanto sugli stessi (scenario_id,k). A validation ci possono essere
# anche varianti MF/GTE: conserviamo GTE selezionato e Popularity, MF è opzionale.
comparison=base_rank.loc[base_rank.model.isin(['popularity','gte_direct','mf_explicit_foldin'])|
                         (base_rank.model==f'mf_foldreg_{float(config["mf_fold_reg_selected"]):g}')].copy()
comparison.loc[comparison.model.str.startswith('mf_foldreg_'),'model']='mf_explicit_foldin'
comparison=pd.concat([comparison,llm_rank],ignore_index=True)
counts=comparison.groupby(['scenario_id','k','model']).size()
assert counts.ge(TOP_K).all()
assert counts.index.get_level_values('model').isin(['popularity','gte_direct','mf_explicit_foldin','llm_gte_semantic']).all()
metrics=evaluate(comparison)
metric_cols=['ndcg@10','recall@10','hr@10','n_unjudged_topk']
per_user=metrics.groupby(['split','userId','model','scheme','k'],as_index=False)[metric_cols].mean()
macro=per_user.groupby(['split','model','scheme','k'],as_index=False)[metric_cols].mean()
display(macro.loc[macro.scheme=='A',['model','k','ndcg@10','recall@10','hr@10','n_unjudged_topk']].round(4))

,model,k,ndcg@10,recall@10,hr@10,n_unjudged_topk
0,gte_direct,0,0.0438,0.0309,0.3,9.6
1,gte_direct,1,0.0066,0.0067,0.1,9.8
2,gte_direct,3,0.0405,0.0490,0.3,9.2
3,gte_direct,5,0.0452,0.0479,0.4,9.3
4,gte_direct,10,0.0583,0.0500,0.3,9.0
10,llm_gte_semantic,0,0.0438,0.0309,0.3,9.6
11,llm_gte_semantic,1,0.0000,0.0000,0.0,9.9
12,llm_gte_semantic,3,0.0110,0.0077,0.1,9.8
13,llm_gte_semantic,5,0.0416,0.0402,0.3,9.5
14,llm_gte_semantic,10,0.0341,0.0195,0.3,9.6


## 9. Confronto **appaiato** e tracciamento della sperimentazione

A parità di utente, scenario, livello e schema calcoliamo il delta tra LLM+GTE e GTE diretto. **Le incertezze vanno stimate per utente**, non trattando le repliche come osservazioni indipendenti. Nel pilota di 10 utenti, gli intervalli sono soltanto orientativi. Niente scelta di configurazioni sul test già ispezionato nei notebook precedenti.

In [59]:
paired=metrics.loc[metrics.model.isin(['gte_direct','llm_gte_semantic'])].pivot_table(
    index=['scenario_id','userId','scheme','k'],columns='model',values='ndcg@10').dropna().reset_index()
paired['delta_ndcg10']=paired['llm_gte_semantic']-paired['gte_direct']
paired_user=paired.groupby(['userId','scheme','k'],as_index=False).delta_ndcg10.mean()
rows=[]
rng=np.random.default_rng(RANDOM_SEED)
for (scheme,k),g in paired_user.groupby(['scheme','k']):
    diffs=g.delta_ndcg10.to_numpy(float)
    if len(diffs)>=2:
        boot=np.mean(rng.choice(diffs,size=(1500,len(diffs)),replace=True),axis=1)
        lo,hi=np.quantile(boot,[.025,.975])
    else:lo=hi=np.nan
    rows.append({'scheme':scheme,'k':int(k),'n_users':len(diffs),'delta_ndcg10':float(diffs.mean()),
                 'ci95_low':float(lo),'ci95_high':float(hi)})
paired_summary=pd.DataFrame(rows)
display(paired_summary.round(4))

# Conteggio costi/latency; valori nulli se il provider non comunica usage.
traces=[]
for p in profiles.to_dict('records'):
    t=p.get('trace')
    if not isinstance(t,dict):continue
    u=t.get('usage') or {}
    traces.append({'scenario_id':p['scenario_id'],'k':p['k'],'provider':t.get('provider'),
                   'model':t.get('model'),'duration_ms':t.get('duration_ms'),
                   'input_tokens':u.get('input_tokens'),'output_tokens':u.get('output_tokens'),
                   'total_tokens':u.get('total_tokens')})
usage=pd.DataFrame(traces)
print('Call traces registrate:',len(usage),'| durata totale ms:',
      round(float(usage.duration_ms.sum()),1) if not usage.empty else 0)

metrics.to_csv(RUN_DIR/'metrics_per_scenario_A_B.csv',index=False)
per_user.to_csv(RUN_DIR/'metrics_per_user_A_B.csv',index=False)
macro.to_csv(RUN_DIR/'macro_A_B.csv',index=False)
paired.to_csv(RUN_DIR/'paired_per_scenario.csv',index=False)
paired_summary.to_csv(RUN_DIR/'paired_delta_ndcg10.csv',index=False)
usage.to_csv(RUN_DIR/'llm_call_usage.csv',index=False)
run_manifest={'protocol_id':PROTOCOL_ID,'run_mode':RUN_MODE,'validation_or_test':selected_split,
              'scenario_count':len(selected_scenarios),'user_count':int(selected_scenarios.userId.nunique()),
              'k_levels':list(K_LEVELS),'prompt_version':PROMPT_VERSION,'prompt_sha256':PROMPT_HASH,
              'provider_profile':PROFILE_MODEL,'embedding_model_query':GTE_MODEL_ID,
              'qdrant_collection':QDRANT_COLLECTION,'qdrant_movie_id_key':QDRANT_MOVIE_ID_KEY,
              'gte_negative_alpha':ALPHA,'negative_only_alpha':NEG_ONLY_ALPHA,
              'popularity_prior':POPULARITY_PRIOR,'movie_embeddings_recomputed':False,
              'baseline_file':str(base_file.relative_to(ROOT)),
              'methodological_limitations':[
                  'MovieLens user tags may already be contained in saved Qdrant movie vectors',
                  'Unobserved ratings are not negatives; offline exposure bias remains',
                  'LLM sees public title/genre, direct GTE uses full indexed movie texts',
                  'No prompt/model tuning permitted using test results',
                  'Observed k=0 fallback is intentionally identical to Popularity']}
(RUN_DIR/'run_manifest.json').write_text(json.dumps(run_manifest,indent=2,ensure_ascii=False),encoding='utf-8')
print('Output salvati in:',RUN_DIR)

,scheme,k,n_users,delta_ndcg10,ci95_low,ci95_high
0,A,0,10,0.0000,0.0000,0.0000
1,A,1,10,-0.0066,-0.0199,0.0000
2,A,3,10,-0.0295,-0.0607,0.0000
3,A,5,10,-0.0036,-0.0392,0.0394
4,A,10,10,-0.0241,-0.0785,0.0245
5,B,0,10,0.0000,0.0000,0.0000
6,B,1,10,-0.0653,-0.1682,0.0075
7,B,3,10,-0.1509,-0.2407,-0.0725
8,B,5,10,-0.1290,-0.2147,-0.0513
9,B,10,10,-0.0968,-0.1745,-0.0246


Call traces registrate: 38 | durata totale ms: 119436.4
Output salvati in: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1/evaluations/llm_gte_phase2a/pilot_validation


## 10. Lettura dei risultati e prossimi passi

- **Pilota**: controlla 5–10 profili, eventuali risposte non fondate, query in inglese, similarità self-check, ranking escludendo le sole interazioni visibili, metriche A/B e costi. Il delta su 10 utenti non costituisce conclusione statistica.
- **Validazione completa**: imposta `RUN_MODE='full_validation'`, alza `MAX_NEW_LLM_CALLS` consapevolmente; conserva prompt/encoder/scoring invariati, oppure registra una nuova versione del prompt e confronta solo in validation.
- **Test**: prima congela configurazione e prompt; poi `RUN_MODE='test'` e `ALLOW_TEST_RUN=True`. Poiché il test storico è già stato esaminato in altre analisi, va dichiarato *esplorativo* se viene ulteriormente usato per scelte di design. Per un test realmente incontaminato, servirà un nuovo split user-disjoint o cross-validation.
- **Fase 2B**: solo dopo 2A entra il simulatore conversazionale che può scegliere quali rating rivelare. Questo notebook NON simula utenti né valuta turni di dialogo.
- **Controllo importante:** l'LLM deve influire sulla query semantica, non introdurre reranking o filtri hard: altrimenti non misureremmo soltanto l'interpretazione del profilo.

**Non confrontare direttamente valori assoluti A vs B:** il numero di giudizi rilevanti differisce. Confronta i modelli *all'interno* di ciascuno schema e sullo stesso insieme di scenari.